In [1]:
# !rm -rf /kaggle/working/Real-ESRGAN
# !git clone --depth 1 https://github.com/aksjfds/Real-ESRGAN.git /kaggle/working/Real-ESRGAN
%pip install -q -r requirements.txt
!python -m py_compile realesrgan.py realesrgan_fast.py realesrgan_fast_entry.py enhance/*.py
!python realesrgan_fast_entry.py --help >/dev/null


[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
INPUT_VIDEO = "cm_4.mp4"
OUTPUT_VIDEO = "realesrgan_basicvsrpp.mp4"

MODEL = "realesr-animevideov3"
MODEL_PATH = ""
SCALE = 2
FPS = "source"

START_TIME = 3 * 60 + 15
TEST_SECONDS = 10
PROGRESS_INTERVAL = 60.0

# BasicVSR++ compressed-video enhancement runs before Real-ESRGAN.
BASICVSRPP = True
BASICVSRPP_TRACK = 1             # 1=fidelity; 2=perceptual; 3=fixed-bitrate fidelity
BASICVSRPP_MODEL_PATH = ""       # empty: download the official checkpoint
BASICVSRPP_GPU = 0               # RTX 4090 single GPU
BASICVSRPP_FP16 = True
BASICVSRPP_CLIP_LENGTH = 9        # longer context; automatically falls back to previous 7 on OOM
BASICVSRPP_CLIP_OVERLAP = 2       # 9/2 gives temporal hop 5 instead of 3
BASICVSRPP_TILE_SIZE = 512        # quality baseline; runtime only selects tiles >=512
BASICVSRPP_TILE_PAD = 32
BASICVSRPP_STRENGTH = 1.0
BASICVSRPP_SCENE_THRESHOLD = 0.30

# Real-ESRGAN shared-memory auto tuning for one RTX 4090.
AUTO_TILE = True
MAX_TILE_SIZE = 1536              # 24 GiB 4090 cap; automatic OOM fallback remains enabled
AUTO_BATCH = True
MAX_BATCH_SIZE = 32               # probed automatically on the single GPU
TILE_SIZE = 256                   # fallback when auto tile is disabled
TILE_PAD = 10
TILE_VERIFY_COVERAGE = False      # debug-only check; disabling does not change pixels
BATCH_SIZE = 4                    # fallback when auto batch is disabled
GPU_IDS = "0"                     # one Real-ESRGAN worker on RTX 4090

COLOR_POLICY = "preserve"        # preserve / bt709
HDR_POLICY = "reject"            # reject / passthrough

VIDEO_CODEC = "libx265"
OUTPUT_PIX_FMT = "auto"
CRF = 18
PRESET = "medium"
CQ = 18
NVENC_PRESET = "p7"
ENCODE_GPU = 0
AUDIO_CODEC = "copy"
AUDIO_BITRATE = "192k"


In [5]:
import os
import shlex
import subprocess
import sys

command = [
    sys.executable, "realesrgan_fast_entry.py",
    "--input", INPUT_VIDEO, "--output", OUTPUT_VIDEO,
    "--model", MODEL, "--model-path", MODEL_PATH,
    "--scale", str(SCALE), "--fps", FPS,
    "--fp16", "--channels-last",
    "--basicvsrpp" if BASICVSRPP else "--no-basicvsrpp",
    "--basicvsrpp-track", str(BASICVSRPP_TRACK),
    "--basicvsrpp-model-path", BASICVSRPP_MODEL_PATH,
    "--basicvsrpp-gpu", str(BASICVSRPP_GPU),
    "--basicvsrpp-fp16" if BASICVSRPP_FP16 else "--no-basicvsrpp-fp16",
    "--basicvsrpp-clip-length", str(BASICVSRPP_CLIP_LENGTH),
    "--basicvsrpp-clip-overlap", str(BASICVSRPP_CLIP_OVERLAP),
    "--basicvsrpp-tile-size", str(BASICVSRPP_TILE_SIZE),
    "--basicvsrpp-tile-pad", str(BASICVSRPP_TILE_PAD),
    "--basicvsrpp-strength", str(BASICVSRPP_STRENGTH),
    "--basicvsrpp-scene-threshold", str(BASICVSRPP_SCENE_THRESHOLD),
    "--auto-tile" if AUTO_TILE else "--no-auto-tile",
    "--max-tile-size", str(MAX_TILE_SIZE),
    "--auto-batch" if AUTO_BATCH else "--no-auto-batch",
    "--max-batch-size", str(MAX_BATCH_SIZE),
    "--tile-size", str(TILE_SIZE), "--tile-pad", str(TILE_PAD),
    "--tile-verify-coverage" if TILE_VERIFY_COVERAGE else "--no-tile-verify-coverage",
    "--batch-size", str(BATCH_SIZE), "--gpu-ids", GPU_IDS,
    "--color-policy", COLOR_POLICY, "--hdr-policy", HDR_POLICY,
    "--video-codec", VIDEO_CODEC, "--output-pix-fmt", OUTPUT_PIX_FMT,
    "--crf", str(CRF), "--preset", PRESET, "--cq", str(CQ),
    "--nvenc-preset", NVENC_PRESET, "--encode-gpu", str(ENCODE_GPU),
    "--audio-codec", AUDIO_CODEC, "--audio-bitrate", AUDIO_BITRATE,
    "--start-time", str(START_TIME), "--test-seconds", str(TEST_SECONDS),
    "--progress-interval", str(PROGRESS_INTERVAL),
    "--ffmpeg-bin", "ffmpeg", "--ffprobe-bin", "ffprobe",
]
print("[command]", shlex.join(command), flush=True)

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"
subprocess.run(command, check=True, env=env)


[command] /usr/local/bin/python realesrgan_fast_entry.py --input cm_4.mp4 --output realesrgan_basicvsrpp.mp4 --model realesr-animevideov3 --model-path '' --scale 2 --fps source --fp16 --channels-last --basicvsrpp --basicvsrpp-track 1 --basicvsrpp-model-path '' --basicvsrpp-gpu 0 --basicvsrpp-fp16 --basicvsrpp-clip-length 9 --basicvsrpp-clip-overlap 2 --basicvsrpp-tile-size 512 --basicvsrpp-tile-pad 32 --basicvsrpp-strength 1.0 --basicvsrpp-scene-threshold 0.3 --auto-tile --max-tile-size 1536 --auto-batch --max-batch-size 32 --tile-size 256 --tile-pad 10 --no-tile-verify-coverage --batch-size 4 --gpu-ids 0 --color-policy preserve --hdr-policy reject --video-codec libx265 --output-pix-fmt auto --crf 18 --preset medium --cq 18 --nvenc-preset p7 --encode-gpu 0 --audio-codec copy --audio-bitrate 192k --start-time 195 --test-seconds 10 --progress-interval 60.0 --ffmpeg-bin ffmpeg --ffprobe-bin ffprobe
[encoder] runtime OK: libx265, RGB48 -> 3840x2160/yuv420p10le
[model] downloading https://g

Real-ESRGAN:   0%|          | 0/250 [00:00<?, ?frame/s]

[basicvsrpp auto] clip=9, tile=768, tiles=6, context_ratio=1.276, gpu0:ok/1.92s/peak=12.86GiB
[basicvsrpp auto] selected clip=9, overlap=2, tile=768, baseline_tile=512, fallback_to_previous_clip=False, autotune=2.1s
[shared-memory] input=shm:23.7MiB, output=shm:189.8MiB, dtype=<class 'numpy.float16'>
[auto-tile] selected=1280, gpu0=0.335s/free=21.84GiB
[auto-batch] tile=1280, batch=16, gpu=0


Real-ESRGAN: 100%|██████████| 250/250 [06:59<00:00,  1.68s/frame, fps=0.574]


[basicvsrpp] clips=50, tiles=300, scene_cuts=0
[basicvsrpp auto] attempts=1, selected_clip=9, selected_tile=768, context_ratio=1.276, autotune=2.1s
[basicvsrpp timing] parallel_wall=113.9s, model_gpu_sum=108.3s, gpu_blend_sum=0.1s, d2h=5.5s, cpu_reduce=3.9s
[range] actual_start=00:03:15.000, actual_end=00:03:25.000, processed_inference_frames=250, output_frames=250, output_duration=10.000s
[run] wall_end=2026-08-06T09:36:51+08:00, elapsed=439.2s, average=0.569 frame/s
[timing] model_startup=3.8s, basicvsrpp_startup=12.0s, decode=1.2s, basicvsrpp=118.4s, inference=114.4s, realesrgan=0.0s, lanczos=9.9s, tile_crop_stitch=77.6s, float_to_rgb48=38.4s, encode_flush=2.8s, audio_mux=0.1s, realesrgan_h2d=0.8s, realesrgan_model_gpu=11.0s, realesrgan_d2h=86.4s, realesrgan_shared_write=12.1s
[size] 3.53 MiB, average_bitrate=2.96 Mb/s
[output] /code/Real-ESRGAN/realesrgan_basicvsrpp.mp4


CompletedProcess(args=['/usr/local/bin/python', 'realesrgan_fast_entry.py', '--input', 'cm_4.mp4', '--output', 'realesrgan_basicvsrpp.mp4', '--model', 'realesr-animevideov3', '--model-path', '', '--scale', '2', '--fps', 'source', '--fp16', '--channels-last', '--basicvsrpp', '--basicvsrpp-track', '1', '--basicvsrpp-model-path', '', '--basicvsrpp-gpu', '0', '--basicvsrpp-fp16', '--basicvsrpp-clip-length', '9', '--basicvsrpp-clip-overlap', '2', '--basicvsrpp-tile-size', '512', '--basicvsrpp-tile-pad', '32', '--basicvsrpp-strength', '1.0', '--basicvsrpp-scene-threshold', '0.3', '--auto-tile', '--max-tile-size', '1536', '--auto-batch', '--max-batch-size', '32', '--tile-size', '256', '--tile-pad', '10', '--no-tile-verify-coverage', '--batch-size', '4', '--gpu-ids', '0', '--color-policy', 'preserve', '--hdr-policy', 'reject', '--video-codec', 'libx265', '--output-pix-fmt', 'auto', '--crf', '18', '--preset', 'medium', '--cq', '18', '--nvenc-preset', 'p7', '--encode-gpu', '0', '--audio-codec', 